# Aula 4, BERT

Notebook executável que acompanha a aula [04-bert.md](../../lessons/modulo-06-transformers/04-bert.md).

## O que vamos fazer aqui

O BERT lê a frase nos dois sentidos e aprende preenchendo lacunas. Vamos demonstrar, com um
modelo simples, por que olhar os dois lados ajuda: prevemos uma palavra mascarada usando só a
esquerda e usando os dois lados, e comparamos. Há ainda um caminho opcional com um BERT de
verdade.

## Contagens do corpus

Montamos contagens de quais palavras seguem quais, a partir de um pequeno corpus. Será a base
das duas formas de previsão.

In [ ]:
from collections import Counter

corpus = [
    "o gato bebe leite", "o cachorro bebe agua",
    "o gato come racao", "o cachorro come osso",
    "o gato bebe agua", "o cachorro come racao",
]
tokens = [s.split() for s in corpus]

prox = Counter()
vocab = set()
for t in tokens:
    vocab.update(t)
    for a, b in zip(t, t[1:]):
        prox[(a, b)] += 1

print("vocabulário:", sorted(vocab))

## Lacuna: "o gato ____ leite"

Comparamos a previsão usando só a palavra à esquerda (estilo esquerda-para-direita) com a que
usa as duas vizinhas (estilo BERT). A esquerda é gato, a direita é leite.

In [ ]:
esquerda, direita = "gato", "leite"


def score_esquerda(w):
    return prox[(esquerda, w)]


def score_bidirecional(w):
    return prox[(esquerda, w)] * prox[(w, direita)]


print("Só esquerda (estilo esquerda-para-direita):")
for w in sorted(vocab, key=score_esquerda, reverse=True)[:3]:
    print(f"  {w:8} {score_esquerda(w)}")

print("\nBidirecional (estilo BERT):")
for w in sorted(vocab, key=score_bidirecional, reverse=True)[:3]:
    print(f"  {w:8} {score_bidirecional(w)}")

Só com a esquerda, bebe e come aparecem como candidatos, ambos plausíveis após gato,
deixando a escolha ambígua. Com o contexto bidirecional, come é eliminado (a sequência come
leite nunca ocorre) e bebe vence. É, em miniatura, o motivo da força do BERT: o contexto dos
dois lados desfaz ambiguidades.

## Caminho opcional, um BERT de verdade

Esta célula usa a biblioteca transformers para preencher a lacuna com um BERT pré-treinado em
português. Roda apenas se a biblioteca e o modelo estiverem disponíveis.

In [ ]:
try:
    from transformers import pipeline

    preenchedor = pipeline("fill-mask", model="neuralmind/bert-base-portuguese-cased")
    for r in preenchedor("O gato [MASK] leite.")[:3]:
        print(f"  {r['token_str']:12} score={r['score']:.3f}")
except Exception as erro:
    print("transformers/modelo não disponível, ficamos com a demonstração acima.")
    print("Para instalar: pip install transformers torch. Detalhe:", erro)

Com um BERT de verdade, as sugestões para a lacuna ficam muito mais ricas e
contextualizadas. Para o projeto, mostre um caso em que o contexto bidirecional acerta e o de
só uma direção fica ambíguo.